Premier League Match Outcome Prediction
CET313 Artificial Intelligence Prototype Development

Notebook 01: Data Loading and Initial Cleaning

This notebook covers data loading, inspection, and basic cleaning for the Premier League match dataset. 
This corresponds to collect and prepare data  of the assignment.

In [1]:
#import libraries
import pandas as pd
import numpy as np


In [2]:
#load dataset
data = pd.read_csv("England CSV.csv")
data.head()


,Date,Season,HomeTeam,AwayTeam,FTH Goals,FTA Goals,FT Result,HTH Goals,HTA Goals,HT Result,...,H Fouls,A Fouls,H Corners,A Corners,H Yellow,A Yellow,H Red,A Red,Display_Order,League
0,16/01/2025,2024/25,Ipswich Town,Brighton & Hove Albion,0,2,A,0.0,1.0,A,...,13.0,14.0,1.0,9.0,2.0,2.0,0.0,0.0,20250116,Premier League
1,16/01/2025,2024/25,Man United,Southampton,3,1,H,0.0,1.0,A,...,7.0,10.0,4.0,4.0,1.0,3.0,0.0,0.0,20250116,Premier League
2,15/01/2025,2024/25,Everton,Aston Villa,0,1,A,0.0,0.0,D,...,17.0,10.0,8.0,5.0,2.0,1.0,0.0,0.0,20250115,Premier League
3,15/01/2025,2024/25,Leicester,Crystal Palace,0,2,A,0.0,0.0,D,...,7.0,6.0,4.0,3.0,0.0,0.0,0.0,0.0,20250115,Premier League
4,15/01/2025,2024/25,Newcastle,Wolves,3,0,H,1.0,0.0,H,...,10.0,13.0,4.0,2.0,0.0,2.0,0.0,0.0,20250115,Premier League


 Dataset Overview

The dataset contains historical English Premier League match data, including teams, match statistics (shots, fouls, corners, cards), and full-time results. The data is sourced from Kaggle and is suitable for supervised machine learning classification.


In [3]:
#dataset and information
print("Dataset shape:", data.shape)
data.info()


Dataset shape: (12153, 25)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12153 entries, 0 to 12152
Data columns (total 25 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Date           12153 non-null  object 
 1   Season         12153 non-null  object 
 2   HomeTeam       12153 non-null  object 
 3   AwayTeam       12153 non-null  object 
 4   FTH Goals      12153 non-null  int64  
 5   FTA Goals      12153 non-null  int64  
 6   FT Result      12153 non-null  object 
 7   HTH Goals      11229 non-null  float64
 8   HTA Goals      11229 non-null  float64
 9   HT Result      11229 non-null  object 
 10  Referee        9329 non-null   object 
 11  H Shots        9329 non-null   float64
 12  A Shots        9329 non-null   float64
 13  H SOT          9329 non-null   float64
 14  A SOT          9329 non-null   float64
 15  H Fouls        9329 non-null   float64
 16  A Fouls        9329 non-null   float64
 17  H Corners      9329 non

In [4]:
#missing value Checks
data.isnull().sum().sort_values(ascending=False)


A Shots          2824
H Corners        2824
H Fouls          2824
A SOT            2824
H SOT            2824
A Corners        2824
H Shots          2824
Referee          2824
A Fouls          2824
H Yellow         2824
A Yellow         2824
H Red            2824
A Red            2824
HT Result         924
HTH Goals         924
HTA Goals         924
Display_Order       0
Date                0
Season              0
FT Result           0
FTA Goals           0
FTH Goals           0
AwayTeam            0
HomeTeam            0
League              0
dtype: int64

 Missing Value Handling

Some match statistics (shots, fouls, corners, and cards) contain missing values, mainly from earlier seasons where these metrics were not recorded.  
To ensure consistency and retain sufficient data, missing numerical values are handled using median imputation.  
Half-time statistics are removed to avoid potential data leakage and simplify the model.


In [5]:
#drop half time columns
# Drop half-time related columns (to avoid leakage)
data = data.drop(columns=["HT Result", "HTH Goals", "HTA Goals"], errors="ignore")


In [6]:
#Identify numeric columns with missing values
numeric_cols = data.select_dtypes(include=["int64", "float64"]).columns
numeric_cols


Index(['FTH Goals', 'FTA Goals', 'H Shots', 'A Shots', 'H SOT', 'A SOT',
       'H Fouls', 'A Fouls', 'H Corners', 'A Corners', 'H Yellow', 'A Yellow',
       'H Red', 'A Red', 'Display_Order'],
      dtype='object')

In [7]:
#Median imputation
# 1) Drop Referee (too many missing values and not useful for prediction here)
data = data.drop(columns=["Referee"], errors="ignore")

# 2) Median-impute numeric columns (no chained assignment)
numeric_cols = data.select_dtypes(include=["int64", "float64"]).columns
data[numeric_cols] = data[numeric_cols].fillna(data[numeric_cols].median())

# 3) Check remaining missing values
data.isnull().sum().sort_values(ascending=False).head(10)



Date             0
H Fouls          0
Display_Order    0
A Red            0
H Red            0
A Yellow         0
H Yellow         0
A Corners        0
H Corners        0
A Fouls          0
dtype: int64

Handling Referee Column
The Referee column contains a large number of missing values and is not essential for match outcome prediction in this prototype.  
To keep the model robust and the preprocessing simple, this column was removed.


In [8]:
#creating target
# Binary target: Home Win = 1, Not Home Win (Draw or Away Win) = 0
data["home_win"] = data["FT Result"].apply(lambda x: 1 if x == "H" else 0)

data["home_win"].value_counts()


home_win
0    6590
1    5563
Name: count, dtype: int64

Target Variable
The prediction task is framed as a binary classification problem:
Home Win (1) if full-time result = Home  
Not Home Win (0) if full-time result = D or A  
This allows evaluation using accuracy, precision, recall, F1-score, and confusion matrix.


In [9]:
data.to_csv("epl_cleaned.csv", index=False)
